```
## Notebook: GraphSAGE_Data001.01_superfund_nodes_v1.ipynb
---  
#### Purpose
- Exclude nonfeature columns
- Clean raw superfund data
- Explode suboptimal string columns ['cat_code', cas_number'] to feature matrices
- Encode where required
- Create ready-to-use superfund nodes

#### Input
- GNN328_county_nodes.csv (raw county data)
- GNN328_super_feats.csv (1718 observations, 24 columns, 1 site_id per superfund site,   
  ['cat_code'] = delimited string of 1 - several values,   
  [cas_number'] = list of strings of 1 - several values)
- GNN328_super_gdf.gpkg

#### Processing

#### Output
- superfund_feature_matrix.csv
- superfund_nodes.csv
- superfund_edge_index.csv

#### Goal: Superfund nodes, county nodes, river nodes, modeling environmental hazards to analyze impact on cancer rate.
```

```
# ───────────────────────────────────────────────────────────────────────
# CODE CELL 1.0: ENVIRONMENT AND RUNTIME CONTROL
# ───────────────────────────────────────────────────────────────────────

Note: This notebook is being developed with Python in a Colab notebook using an L4 or A100 hardware accelerator. There are runtime-specific dependencies that must be aligned. Cell 1 should be run prior to starting work in each new runtime. After Cell 1 completes, the runtime must be restarted. Cell 1 may then be commented out to avoid reinstalling the dependencies after subsequent runtime restarts.
```

In [ ]:
# !pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse torch-geometric
# !pip install torch==2.0.0+cu118 torchvision==0.15.1+cu118 torchaudio==2.0.1+cu118 -f https://download.pytorch.org/whl/torch_stable.html
# !pip install torch-scatter torch-sparse torch-geometric -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
# !pip install numpy==1.24.4
# !pip install Optuna

```
# ───────────────────────────────────────────────────────────────────────
# CODE CELL 2.0: IMPORT STATEMENTS
# ───────────────────────────────────────────────────────────────────────
```

In [ ]:
# Torch and torch utilities are for construction of HeteroGRAPH object
# ───────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch_geometric
from torch_geometric.data import HeteroData
from torch.optim import Adam
from torch_geometric.nn import SAGEConv, HeteroConv
import numpy as np


# Sklearn encoder, training and evaluation tools
# ───────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import optuna

# Data handling libraries
# ───────────────────────────────────────────────────────────────────────
from shapely.geometry import Point # For reading geometry in super_gdf
import geopandas as gpd            # For loading super_gdf
import pandas as pd

# Utilities
# ───────────────────────────────────────────────────────────────────────
from google.colab import files
import random
import inspect
import math
import time
import re

# Utilities
# ───────────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import tabulate
from tabulate import tabulate
import collections
from collections import Counter

# Set random state for everything, everywhere, all at once.
# ───────────────────────────────────────────────────────────────────────
def jenny_setter(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

jenny = 8675309
jenny_setter(seed=jenny)

# Check runtime dependency control. Confirm correct PyTorch, CUDA, numpy.
# ───────────────────────────────────────────────────────────────────────
rows = [
        ["Torch", "Torch 2.0.0", torch.__version__],
        ["CUDA_version", "11.8", torch.version.cuda],
        ["NumPy", "NumPy 1.24.4", np.__version__]
       ]
print(tabulate(rows, headers=["Library", "Expected Version", "Current Version"], tablefmt="grid"))

+--------------+--------------------+-------------------+
| Library      | Expected Version   | Current Version   |
+==============+====================+===================+
| Torch        | Torch 2.0.0        | 2.0.0+cu118       |
+--------------+--------------------+-------------------+
| CUDA_version | 11.8               | 11.8              |
+--------------+--------------------+-------------------+
| NumPy        | NumPy 1.24.4       | 1.24.4            |
+--------------+--------------------+-------------------+


```
# ───────────────────────────────────────────────────────────────────────
# CELL 3.0: LOAD DATA, PREPROCESSING
# ───────────────────────────────────────────────────────────────────────
```

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# CELL 3.1: LOAD DATA, PREPROCESSING
# ───────────────────────────────────────────────────────────────────────

# county_df is the county data set.
# RangeIndex: 2930 entries, 0 to 2929, 3140 unique values in 'fips.
# None of the columns start out with the right data type :P
# county_df.isna().sum().sum() == 0
# ──────────────────────────────────────────────────────────────────────────────
county_df = gpd.read_file('/content/GNN328_county_nodes.csv')

# >>> fips is currently object dtype ensure <<<
# >>> it's correctly string and 5-padded.   <<<
# ──────────────────────────────────────────────────────────────────────────────
county_df['fips'] = county_df['fips'].astype(str).str.zfill(5)

# >>> deal with data types <<<
# ──────────────────────────────────────────────────────────────────────────────
numeric_ls = ['cancer_rate','avg_annual_count','five_year_trend','has_cancer_data']
string_ls = ['county','state']
categ_ls = ['rural_urban_code','recent_trend']

for col in numeric_ls:
  county_df[col] = pd.to_numeric(county_df[col], errors='coerce')
for col in string_ls:
    county_df[col] = county_df[col].astype(str)
county_df['rural_urban_code'] = county_df['rural_urban_code'].astype('category')
county_df['recent_trend'] = county_df['recent_trend'].astype('category')
county_df = county_df[county_df['has_cancer_data'] == 1].copy()
county_df['five_year_trend'] = county_df['five_year_trend'].fillna(-999)

# ──────────────────────────────────────────────────────────────────────────────
# >>> VALIDATION                                                             <<<
# >>> Length, number of unique fips and unique site_ids should be equal.     <<<
# ──────────────────────────────────────────────────────────────────────────────
print('___Ensure appropriate dtypes applied___')
display(county_df.info())
print('───────────────────────────────────────')
print()
print('_Check for the NAs being reintroduced__')
display(county_df.isna().sum())
print('───────────────────────────────────────')

___Ensure appropriate dtypes applied___
<class 'pandas.core.frame.DataFrame'>
Index: 2930 entries, 0 to 2929
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   county            2930 non-null   object  
 1   state             2930 non-null   object  
 2   fips              2930 non-null   object  
 3   rural_urban_code  2930 non-null   category
 4   cancer_rate       2930 non-null   float64 
 5   avg_annual_count  2930 non-null   float64 
 6   recent_trend      2930 non-null   category
 7   five_year_trend   2930 non-null   float64 
 8   has_cancer_data   2930 non-null   int64   
dtypes: category(2), float64(3), int64(1), object(3)
memory usage: 189.2+ KB


None

───────────────────────────────────────

_Check for the NAs being reintroduced__


,0
county,0
state,0
fips,0
rural_urban_code,0
cancer_rate,0
avg_annual_count,0
recent_trend,0
five_year_trend,0
has_cancer_data,0


───────────────────────────────────────


```
# ───────────────────────────────────────────────────────────────────────
# CELL 4.0: ENCODING: ['rural_urban_code', 'recent_trend']
#           using pd.get_dummies() due to:
#           1. No meaning to distance between values
#           2. Ordinal but not proportional
# ───────────────────────────────────────────────────────────────────────
```

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# CELL 4.1: ENCODING: ['rural_urban_code', 'recent_trend']
# ───────────────────────────────────────────────────────────────────────

county_df = pd.get_dummies(
    county_df,
    columns=['rural_urban_code', 'recent_trend'],
    prefix=['rural', 'trend']
)
# ──────────────────────────────────────────────────────────────────────────────
# >>> VALIDATION (county_df)                                                 <<<
# >>> check NAs == 0, head looks correct, .info() shows right data types     <<<
# ──────────────────────────────────────────────────────────────────────────────
print('__post-encoding county_df.isna().sum()___')
display(county_df.isna().sum())
print('_________________________________________')
print()
print('__post-encoding county_df.head()___')
display(county_df.head())
print('___________________________________')
print()
print('__post-encoding county_df.info()___')
display(county_df.info())

__post-encoding county_df.isna().sum()___


,0
county,0
state,0
fips,0
cancer_rate,0
avg_annual_count,0
five_year_trend,0
has_cancer_data,0
rural_rural,0
rural_urban,0
trend_,0


_________________________________________

__post-encoding county_df.head()___


,county,state,fips,cancer_rate,avg_annual_count,five_year_trend,has_cancer_data,rural_rural,rural_urban,trend_,trend_falling,trend_rising,trend_stable
0,traverse county,minnesota,27155,693.5,37.0,2.2,1,True,False,False,False,False,True
1,polk county,texas,48373,679.5,436.0,-0.3,1,True,False,False,False,False,True
2,galax city,virginia,51640,655.0,55.0,1.7,1,True,False,False,False,False,True
3,greeley county,nebraska,31077,653.1,21.0,0.7,1,True,False,False,False,False,True
4,dewey county,south dakota,46041,634.5,28.0,1.3,1,True,False,False,False,False,True


___________________________________

__post-encoding county_df.info()___
<class 'pandas.core.frame.DataFrame'>
Index: 2930 entries, 0 to 2929
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   county            2930 non-null   object 
 1   state             2930 non-null   object 
 2   fips              2930 non-null   object 
 3   cancer_rate       2930 non-null   float64
 4   avg_annual_count  2930 non-null   float64
 5   five_year_trend   2930 non-null   float64
 6   has_cancer_data   2930 non-null   int64  
 7   rural_rural       2930 non-null   bool   
 8   rural_urban       2930 non-null   bool   
 9   trend_            2930 non-null   bool   
 10  trend_falling     2930 non-null   bool   
 11  trend_rising      2930 non-null   bool   
 12  trend_stable      2930 non-null   bool   
dtypes: bool(6), float64(3), int64(1), object(3)
memory usage: 200.3+ KB


None

```
# ───────────────────────────────────────────────────────────────────────
# CELL 5.0: FREEZE THE FINAL FEATURE MATRIX AND REFERENCE DATAFRAMES
# ───────────────────────────────────────────────────────────────────────
```

In [ ]:
# ──────────────────────────────────────────────────────────────
# CELL 5.1: FREEZE THE FINAL COUNTY FEATURE MATRIX
# ──────────────────────────────────────────────────────────────
county_feature_columns = [
    'cancer_rate', 'avg_annual_count', 'five_year_trend',
    'rural_rural', 'rural_urban', 'trend_',
    'trend_falling', 'trend_rising', 'trend_stable'
]
county_feature_matrix = county_df[county_feature_columns].copy().astype(float)
data200_node_id_map = county_df[['fips']].reset_index().rename(columns={'index': 'row_index'})

display(county_feature_matrix)

,cancer_rate,avg_annual_count,five_year_trend,rural_rural,rural_urban,trend_,trend_falling,trend_rising,trend_stable
0,693.5,37.0,2.2,1.0,0.0,0.0,0.0,0.0,1.0
1,679.5,436.0,-0.3,1.0,0.0,0.0,0.0,0.0,1.0
2,655.0,55.0,1.7,1.0,0.0,0.0,0.0,0.0,1.0
3,653.1,21.0,0.7,1.0,0.0,0.0,0.0,0.0,1.0
4,634.5,28.0,1.3,1.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...
2925,219.1,7.0,-999.0,1.0,0.0,1.0,0.0,0.0,0.0
2926,219.0,6.0,-2.1,0.0,1.0,0.0,0.0,0.0,1.0
2927,208.0,10.0,-2.5,1.0,0.0,0.0,0.0,0.0,1.0
2928,207.4,4.0,-1.6,1.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
# ──────────────────────────────────────────────────────────────
# CELL 5.2: WRITE THE REFERENCE DATA_FRAMES
# ──────────────────────────────────────────────────────────────
county_df.to_csv('DATA002_county_feature_dataframe_v1.csv', index=False, float_format="%.10g")
county_feature_matrix.to_csv('DATA002_county_feature_matrix_v1.csv', index=False, float_format="%.10g")
data200_node_id_map.to_csv('DATA002_county_node_id_map_v1.csv', index=False, float_format="%.10g")

In [ ]:
print(county_df.columns)
print(county_df.info())
print(county_df.isna().sum())

Index(['county', 'state', 'fips', 'cancer_rate', 'avg_annual_count',
       'five_year_trend', 'has_cancer_data', 'rural_rural', 'rural_urban',
       'trend_', 'trend_falling', 'trend_rising', 'trend_stable'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
Index: 2930 entries, 0 to 2929
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   county            2930 non-null   object 
 1   state             2930 non-null   object 
 2   fips              2930 non-null   object 
 3   cancer_rate       2930 non-null   float64
 4   avg_annual_count  2930 non-null   float64
 5   five_year_trend   2930 non-null   float64
 6   has_cancer_data   2930 non-null   int64  
 7   rural_rural       2930 non-null   bool   
 8   rural_urban       2930 non-null   bool   
 9   trend_            2930 non-null   bool   
 10  trend_falling     2930 non-null   bool   
 11  trend_rising      2930 non-null   bool   
 12  tr

In [ ]:
# ───────────────────────────────────────────────────────────────────────
# CELL 999.0: SCRATCH CELL
# ───────────────────────────────────────────────────────────────────────